# 05 Visualize Phi Needed

`04a` と `04b` の出力を読み込み、`A_unit(m_a)`、`R_{\tau,\max}^{unit}(m_a)`、`phi_needed(m_a)` を人間が読みやすい 2 次元グラフで確認する notebook です。

主に使う入力ファイル:

- `results/04a-a-unit/A_unit_scan.csv`
- `results/04b-rtau-needed/Rtau_phi_needed_scan.csv`
        


In [ ]:
from pathlib import Path
import csv
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve().parent
A_UNIT_CSV = ROOT / "results/04a-a-unit/A_unit_scan.csv"
RTAU_CSV = ROOT / "results/04b-rtau-needed/Rtau_phi_needed_scan.csv"


def read_csv(path):
    with path.open() as f:
        return list(csv.DictReader(f))


a_rows = read_csv(A_UNIT_CSV)
r_rows = read_csv(RTAU_CSV)

masses = np.array([float(row["mass_eV"]) for row in a_rows])
A_unit = np.array([float(row["A_unit"]) for row in a_rows])

case_rows = defaultdict(list)
for row in r_rows:
    case_rows[row["case_name"]].append(row)

case_names = sorted(case_rows)
case_names
        


## `A_unit(m_a)` の形

絶対値と符号つきの両方を見ます。  
高質量側で絶対値が plateau に近いか、位相の都合で符号が振れているかを確認できます。
        


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].loglog(masses, np.abs(A_unit))
axes[0].set_xlabel(r"$m_a\ [{\rm eV}]$")
axes[0].set_ylabel(r"$|A_{\rm unit}(m_a)|$")
axes[0].set_title(r"Absolute Unit Response")
axes[0].grid(True, which="both", alpha=0.3)

axes[1].semilogx(masses, A_unit)
axes[1].axhline(0.0, color="black", lw=1.0, alpha=0.6)
axes[1].set_xlabel(r"$m_a\ [{\rm eV}]$")
axes[1].set_ylabel(r"$A_{\rm unit}(m_a)$")
axes[1].set_title(r"Signed Unit Response")
axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()
        


## `R_{\tau,\max}^{unit}(m_a)` の比較

toy `C_L^{\phi\phi}` の仮定ごとに、どの質量帯が最も有利かを比較します。
        


In [ ]:
plt.figure(figsize=(8, 5.5))
for case_name in case_names:
    rows = sorted(case_rows[case_name], key=lambda r: float(r["mass_eV"]))
    mass = np.array([float(row["mass_eV"]) for row in rows])
    rtau = np.array([float(row["Rtau_max_unit"]) for row in rows])
    plt.loglog(mass, rtau, label=case_name)

plt.xlabel(r"$m_a\ [{\rm eV}]$")
plt.ylabel(r"$R_{\tau,\max}^{\rm unit}(m_a)$")
plt.title(r"Maximum Toy Patchy-to-Genuine Ratio")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
        


## `phi_needed(m_a)` の比較

`R_target = 0.1, 1, 10` について、必要な振幅を比較します。  
縦軸が低いほど、その target を達成しやすいことを意味します。
        


In [ ]:
targets = ["0.1", "1", "10"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharex=True)

for ax, target in zip(axes, targets):
    for case_name in case_names:
        rows = sorted(case_rows[case_name], key=lambda r: float(r["mass_eV"]))
        mass = np.array([float(row["mass_eV"]) for row in rows])
        phi_needed = np.array([float(row[f"phi_needed_target_{target}"]) for row in rows])
        ax.loglog(mass, phi_needed, label=case_name)
    ax.set_xlabel(r"$m_a\ [{\rm eV}]$")
    ax.set_ylabel(r"$\phi_{\rm needed}(m_a)$")
    ax.set_title(rf"$R_{{\tau,\max}} = {target}$")
    ax.grid(True, which="both", alpha=0.3)

axes[0].legend()
plt.tight_layout()
plt.show()
        


## best mass の一覧

各 toy case で `R_{\tau,\max}^{unit}` が最大となる質量点を抜き出します。
        


In [ ]:
summary = []
for case_name in case_names:
    rows = case_rows[case_name]
    best = max(rows, key=lambda r: float(r["Rtau_max_unit"]))
    summary.append(
        {
            "case": case_name,
            "m_best_eV": float(best["mass_eV"]),
            "Rtau_max_unit": float(best["Rtau_max_unit"]),
            "L_peak": int(float(best["L_peak"])),
            "phi_needed_R01": float(best["phi_needed_target_0.1"]),
            "phi_needed_R1": float(best["phi_needed_target_1"]),
            "phi_needed_R10": float(best["phi_needed_target_10"]),
        }
    )

summary
        


## best mass に印をつけた図

どの case でも best mass が同じ帯に集まっているかを視覚的に確認します。
        


In [ ]:
plt.figure(figsize=(8, 5.5))
for item in summary:
    rows = sorted(case_rows[item["case"]], key=lambda r: float(r["mass_eV"]))
    mass = np.array([float(row["mass_eV"]) for row in rows])
    rtau = np.array([float(row["Rtau_max_unit"]) for row in rows])
    plt.loglog(mass, rtau, alpha=0.5)
    plt.scatter(item["m_best_eV"], item["Rtau_max_unit"], s=70, label=item["case"])

plt.xlabel(r"$m_a\ [{\rm eV}]$")
plt.ylabel(r"$R_{\tau,\max}^{\rm unit}(m_a)$")
plt.title(r"Best-Mass Locations by Toy Case")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
        
